In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import pandas as pd
from forecast_returns_fixed import forecast_returns

# --- Tiêu đề ---
display(HTML("<h2 style='text-align:center; color:#2E86C1;'>📦 Dự báo số lượng hàng hoàn</h2>"))

# --- Chọn chế độ ---
mode_radio = widgets.RadioButtons(
    options=[("📁 Dự báo từ file CSV", "file"), ("🧮 Nhập dữ liệu thủ công", "manual")],
    value="file",
    description="Chế độ:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="400px")
)

# --- Các widget cho chế độ "file" ---
file_input = widgets.Text(
    value="ecommerce_returns_synthetic_data.csv",
    description="Tên file CSV:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="450px")
)
freq_dropdown = widgets.Dropdown(
    options=[("Ngày", "D"), ("Tuần", "W"), ("Tháng", "M")],
    value="D",
    description="Tần suất:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="300px")
)
period_slider = widgets.IntSlider(
    value=30, min=7, max=180, step=1,
    description="Dự báo (ngày):",
    style={'description_width': 'initial'},
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)

# --- Các widget cho chế độ "manual" ---
date_input = widgets.DatePicker(
    description="Ngày dự đoán:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="300px")
)
returns_input = widgets.FloatText(
    value=0,
    description="Số lượng hoàn (ước tính gần nhất):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="400px")
)

# --- Nút chạy ---
run_button = widgets.Button(
    description="🚀 Chạy dự báo",
    button_style="success",
    layout=widgets.Layout(width="200px", align_self="center")
)

# --- Khu vực hiển thị kết quả ---
output_area = widgets.Output()

# --- Logic khi nhấn nút ---
def on_run_clicked(b):
    with output_area:
        clear_output()
        print("🔄 Đang chạy dự báo...")

        try:
            if mode_radio.value == "file":
                # Chạy dự báo từ file CSV
                forecast_df, model = forecast_returns(
                    input_file=file_input.value,
                    freq=freq_dropdown.value,
                    periods=period_slider.value
                )

                plt.figure(figsize=(12,5))
                plt.plot(forecast_df["ds"], forecast_df["yhat"], label="Dự báo (yhat)")
                plt.fill_between(forecast_df["ds"], forecast_df["yhat_lower"], forecast_df["yhat_upper"], alpha=0.2, label="Khoảng tin cậy")
                if forecast_df["y"].notna().any():
                    plt.plot(forecast_df["ds"], forecast_df["y"], label="Thực tế (y)", marker="o", markersize=3)
                plt.xlabel("Ngày")
                plt.ylabel("Số lượng hàng hoàn")
                plt.title("📈 Biểu đồ dự báo từ file CSV")
                plt.legend()
                plt.tight_layout()
                plt.show()

                display(forecast_df.tail(10))

            else:
                # Chạy dự báo thủ công (ví dụ đơn giản)
                if not date_input.value:
                    print("⚠️ Vui lòng chọn ngày dự đoán.")
                    return

                df = pd.DataFrame([{
                    "ds": pd.to_datetime(date_input.value),
                    "y": returns_input.value
                }])

                print("📅 Dữ liệu nhập tay:")
                display(df)

                # 👉 Ở đây bạn có thể gọi model dự báo của mình
                # giả sử bạn muốn hiển thị kết quả mô phỏng
                predicted = returns_input.value * 1.05  # ví dụ tăng 5%
                print(f"🔮 Dự báo số lượng hàng hoàn ngày {date_input.value}: {predicted:.2f}")

        except Exception as e:
            print("❌ Lỗi khi dự báo:", e)

# --- Gắn sự kiện ---
run_button.on_click(on_run_clicked)

# --- Ẩn/hiện các phần theo chế độ ---
def update_mode(change):
    if change["new"] == "file":
        file_controls.layout.display = "flex"
        manual_controls.layout.display = "none"
    else:
        file_controls.layout.display = "none"
        manual_controls.layout.display = "flex"

mode_radio.observe(update_mode, names="value")

# --- Gom nhóm giao diện ---
file_controls = widgets.VBox([file_input, freq_dropdown, period_slider])
manual_controls = widgets.VBox([date_input, returns_input])
manual_controls.layout.display = "none"

controls = widgets.VBox([
    mode_radio,
    file_controls,
    manual_controls,
    run_button
])

ui = widgets.VBox([controls, output_area], layout=widgets.Layout(align_items='center'))
display(ui)
